In [ ]:
import re
import pandas as pd
from pathlib import Path
from pymongo import MongoClient

In [ ]:
# ======= CONFIG =======
csv_folder = Path('dataset_kula/')
mongo_uri = 'mongodb://localhost:27017'
db_name = 'pipeline_db'

mode = 'overwrite'
csv_pattern = '*.csv'

# ======= HELPERS =======
def safe_collection_name(filename: str) -> str:
    base = Path(filename).stem
    base = re.sub(r"[^a-zA-Z0-9_]+", "_", base).strip("_")
    if not base:
        base = "collection"
    return base [:120]

def sanitize_field_name(col: str) -> str:
    col = str(col)
    col = col.replace('.', '_')
    if col.startswith('$'):
        col = '_' + col[1:]
    return col

def sanitize_dataframe_columns(df: pd.Dataframe) -> pd.DataFrame:
    df = df.copy()
    df.columns = [sanitize_field_name(c) for c in df.columns]
    return df

# ======= MAIN =======
client = MongoClient(mongo_uri)
db = client[db_name]

csv_files = sorted(csv_folder.glob(csv_pattern))
print('Found CSV files:', len(csv_files))

for f in csv_files:
    df = pd.read_csv(f)
    df['_source_file'] = f.name
    df = sanitize_dataframe_columns(df)
    df = df.where(pd.notnull(df), None)
    
    col_name = safe_collection_name(f.name)
    col = db[col_name]

    if mode == 'overwrite':
        col.delete_many({})

    records = df.to_dict(orient='records')
    if records:
        col.insert_many(records)
        print(f"{f.name} -> {db_name}.{col_name}: inserted {len(records)} docs")
    else:
        print(f"{f.name} -> {db_name}.{col_name}: no rows")
print("\nDone.")

In [ ]:
# ----- CONFIG -----
csv_folder = Path("dataset_kula/")
mongo_uri = 'mongodb://127.0.0.0.1:27017'
db_name = 'scrape_pipeline'
mode = 'overwrite'

# ----- HELPERS -----
def safe_collection_name(filename: str) -> str:
    base = Path(filename).stem
    base = re.sub(r"[^a-zA-Z0-9_]+", '_', base).strip('_')
    return (base or 'collection')[:120]

def sanitize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        ('_' + c[1:] if str(c).startwith('$') else str (c)).replace('.', '_') for c in df.columns
    ]
    return df

# ----- CONNECT -----
client = MongoClient(mongo_uri)
db = client[db_name]

# ----- LOAD & INISERT -----
files = sorted(csv_folder.glob('*.csv'))
print('CSV files found:', len(files))

for f in files:
    df = pd.read_csv(f)
    df['_source_file'] = f.name
    df = sanitize_columns(df)
    df = df.where(pd.notnull(df), None)

    col_name = safe_collection_name(f.name)
    col = df[col_name]

    if mode == 'overwrite':
        col.delete_many({})

    records = df.to_dict(orient='records')
    if records:
        col.insert_many(records)
        print(f"{f.name} -> {db_name}.{col_name}: {len(records)} docs")
    else:
        print(f"{f.name} -> {db_name}.{col_name}: empty")

print("Done.")